<a href="https://colab.research.google.com/github/buman143/dspy/blob/main/dspy_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# Run once per session

!pip install dspy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.0/331.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.5/146.5 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.0/17.0 MB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 7.1 MB/s eta 0:00:00
  Attempting uninstall: typeguard
    Found existing installation: typeguard 4.5.1
    Uninstalling typeguard-4.5.1:
      Successfully uninstalled typeguard-4.5.1


In [23]:
# Imports

import dspy
import os

from pprint import pprint
from dotenv import load_dotenv

In [11]:
# Load API key from .env

load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')

In [9]:
# Set up LM

lm = dspy.LM('openai/gpt-4o-mini', api_key=api_key)
dspy.configure(lm=lm)

In [14]:
# Make a test call

lm("Say this is a test!", temperature=0.7)  # => ['This is a test!']
lm(messages=[{"role": "user", "content": "Say this is a test!"}])  # => ['This is a test!']

['This is a test! How can I assist you further?']

In [15]:
# Define a module (ChainOfThought) and assign it a signature (return an answer, given a question).
qa = dspy.ChainOfThought('question -> answer')

# Run with the default LM configured with `dspy.configure` above.
response = qa(question="How many floors are in the Empire State Building?")
print(response.answer)

The Empire State Building has 102 floors.


In [18]:
# Changing model within context

with dspy.context(lm=dspy.LM('openai/gpt-3.5-turbo')):
    response = qa(question="How many floors are in the Empire State Building?")
    print(f'GPT-3.5-turbo:\t{response.answer}')

GPT-3.5-turbo:	102


In [24]:
# Output and usage metadata using lm.history

print(f'Calls:\t{len(lm.history)}')

pprint(lm.history[0])

Calls:	3
{'cost': 9.149999999999999e-06,
 'kwargs': {'temperature': 0.7},
 'messages': None,
 'model': 'openai/gpt-4o-mini',
 'model_type': 'chat',
 'outputs': ['This is a test! How can I assist you today?'],
 'prompt': 'Say this is a test!',
 'response': ModelResponse(id='chatcmpl-DihGxvqQhptdy31wPGiDJyR7rs4Ma', created=1779545143, model='gpt-4o-mini-2024-07-18', object='chat.completion', system_fingerprint='fp_9cdcdc636e', choices=[Choices(finish_reason='stop', index=0, message=Message(content='This is a test! How can I assist you today?', role='assistant', tool_calls=None, function_call=None, provider_specific_fields={'refusal': None}, annotations=[]), provider_specific_fields={})], usage=Usage(completion_tokens=12, prompt_tokens=13, total_tokens=25, completion_tokens_details=CompletionTokensDetailsWrapper(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0, text_tokens=None, image_tokens=None, video_tokens=None), prompt_tokens_details=Prom

In [28]:
# Signatures

# Sentiment classification example
toxicity = dspy.Predict(
    dspy.Signature(
        "text -> toxic: bool",
        instructions="Mark as 'toxic' if the text contains insults, harassment, or derogatory remarks."
    )
)

for text in ['You are beautiful.', 'You are dumb.']:

    # The passed kwarg must match the toxicity obj's Signature var.
    # Ex. text is the input var and toxic is the output var
    print(f'{text} -> {toxicity(text=text).toxic}')

You are beautiful. -> False
You are dumb. -> True


In [ ]:
# Summarization example

